In [15]:
import sys
import pandas as pd

print("Python utilisé :", sys.executable)
print("Version de pandas :", pd.__version__)

Python utilisé : c:\Users\Utilisateur\Documents\brief-tva\Brief-TVA-\.venv\Scripts\python.exe
Version de pandas : 3.1.1


In [16]:
from pathlib import Path

# Repérer le dossier du projet
racine = Path.cwd()

if not (racine / "starter-kit").exists():
    racine = racine.parent

dossier_data = racine / "starter-kit" / "data"

print("Dossier des données :", dossier_data)
print("Le dossier existe :", dossier_data.exists())
print("Fichiers présents :")

for fichier in dossier_data.iterdir():
    print("-", fichier.name)


Dossier des données : c:\Users\Utilisateur\Documents\brief-tva\Brief-TVA-\starter-kit\data
Le dossier existe : True
Fichiers présents :
- numeros_tva.csv
- numeros_tva.xlsx


In [17]:
df = pd.read_csv(
    dossier_data / "numeros_tva.csv",
    dtype=str,
    keep_default_na=False
)

Phase 1 — Explorer le référentiel

Point 1 Combien de pays distincts ? Combien de formats différents pour un même pays ?

In [18]:
# Compter les lignes pour chaque code pays, sans modifier les données
effectifs_pays = df["pays_declare"].value_counts(dropna=False)

print("Nombre de codes distincts :", len(effectifs_pays))
print("\nNombre de lignes par code :")
print(effectifs_pays.to_string())
print("\nTotal des lignes :", effectifs_pays.sum())

Nombre de codes distincts : 15

Nombre de lignes par code :
pays_declare
FR    974
DK    961
BE    959
LU    956
SE    949
PT    947
NL    947
IT    947
PL    939
FI    902
ZZ    115
QQ    109
GB    104
UK    104
XX     87

Total des lignes : 10000


Combien de formats différents pour un même pays ?

On choisit le pays France (FR). On affiche simplement les numéros associés au pays FR, sans tirage au hasard ni tableau supplémentaire.

In [19]:
display(df.loc[df["pays_declare"] == "FR", ["numero_tva"]].head(30))

,numero_tva
19,FR63432337650
22,FR58658124710
27,FR14760307373
29,FR84248701951
51,FR82738452940
57,FR.76418167474
58,FR02831068190
91,
100,FR27552032534
103,FR53917821806


Cet aperçu montre déjà plusieurs présentations pour le pays déclaré FR :
- Préfixe collé aux chiffres : FR63432337650.
- Point après le préfixe : FR.76418167474.
- Préfixe absent : 58560322811.
- Lettre dans la partie après le préfixe : FRH3842670445.
- Tiret après le préfixe : FR-38430871616.
- Une valeur visuellement vide à l’index 91.


On regarde maintenant les longeurs sur toutes les lignes déclarées FR.

In [20]:
df[df["pays_declare"] == "FR"]["numero_tva"].str.len().value_counts()

numero_tva
13    624
14    142
16     56
11     47
15     46
12     32
1      10
4       7
3       5
0       4
17      1
Name: count, dtype: int64

Le résultat montre 11 longeurs différentes pour 974 lignes déclarées FR. La plus fréquente est 13 caractères : 624 lignes. 
Attention : 11 longueurs ne veut pas dire 11 formats. Deux textes de même longeur peuvent avoir des présentations différentes. 

Regardons maintenant les valeurs de 14 caractères, la deuxième longueur la plus fréquente :

In [21]:
# Sélectionner les numéros du pays déclaré FR
tva_fr = df[df["pays_declare"] == "FR"]["numero_tva"]

# Afficher les 10 premiers numéros de 14 caractères
display(tva_fr[tva_fr.str.len() == 14].head(10))

57     FR.76418167474
129    FR568210454899
169    FR-38430871616
279    FR305495935677
364    FR.44633914443
405    FR.03158245118
491    FR941072657962
539    FR003433647676
567    FR.16922728668
636    FR.87411489351
Name: numero_tva, dtype: str

On voit déjà plusieurs présentations parmi les valeurs de 14 caractères :
- Avec un point : FR.76418167474.
- Avec un tiret : FR-38430871616.
- Sans séparateur visible : FR568210454899.
Pour les dernières, un espace invisible peut aussi expliquer la longueur.

In [22]:
# Rendre visibles les espaces dans les mêmes exemples
display(tva_fr[tva_fr.str.len() == 14].head(10).map(repr))

57     'FR.76418167474'
129    'FR568210454899'
169    'FR-38430871616'
279    'FR305495935677'
364    'FR.44633914443'
405    'FR.03158245118'
491    'FR941072657962'
539    'FR003433647676'
567    'FR.16922728668'
636    'FR.87411489351'
Name: numero_tva, dtype: str

Dans ces 10 exemples, aucun espace au début ou à la fin n’apparaît. On distingue :
- FR. suivi de chiffres ;
- FR- suivi de chiffres ;
- FR suivi de 12 chiffres, sans séparateur.
Pour compter les présentations sur toutes les lignes FR, remplaçons chaque chiffre par 0 dans un aperçu :

In [23]:
formats_fr = tva_fr.str.replace(r"[0-9]", "0", regex=True)

print(formats_fr.value_counts().to_string())

numero_tva
FR00000000000        525
  FR00000000000       56
FR-00000000000        49
00000000000           47
FR.00000000000        46
FR000000000000        46
FR 0000 0000000       46
fr00000000000         44
FR0000000000          32
null                   7
N/A                    5
                       5
-                      5
                       4
FR000M0000000          3
FR0000000000K          3
FR0K000000000          3
FR00000000H00          3
FR00000000J00          3
FRH0000000000          2
FR0000000O000          2
FR000H0000000          2
FR0J000000000          2
FR0000O000000          2
FR0000000000O          2
FR000000000G0          2
FRM0000000000          2
FR00000000K00          2
FR00O00000000          2
FR000000000M0          2
FR0000000G000          1
FR000K0000000          1
FR000000O0000          1
FR00000O00000          1
FR0000H000000          1
FR00G00000000          1
FR000000K0000          1
FR0000G000000          1
FR000000000O0          1
FR000000000J0 

Le regroupement fonctionne. Il montre notamment des numéros avec tiret, point, espaces, préfixe en minuscules ou lettres à l’intérieur.

Mais l’affichage est tronqué : on ne voit pas toutes les formes.

Pour obtenir leur nombre et rendre les espaces visibles :

In [24]:
print("Nombre de formes observées :", formats_fr.nunique())

display(formats_fr.map(repr).value_counts())

Nombre de formes observées : 50


numero_tva
'FR00000000000'        525
'  FR00000000000 '      56
'FR-00000000000'        49
'00000000000'           47
'FR.00000000000'        46
'FR000000000000'        46
'FR 0000 0000000'       46
'fr00000000000'         44
'FR0000000000'          32
'null'                   7
'N/A'                    5
' '                      5
'-'                      5
''                       4
'FR000M0000000'          3
'FR0000000000K'          3
'FR0K000000000'          3
'FR00000000H00'          3
'FR00000000J00'          3
'FRH0000000000'          2
'FR0000000O000'          2
'FR000H0000000'          2
'FR0J000000000'          2
'FR0000O000000'          2
'FR0000000000O'          2
'FR000000000G0'          2
'FRM0000000000'          2
'FR00000000K00'          2
'FR00O00000000'          2
'FR000000000M0'          2
'FR0000000G000'          1
'FR000K0000000'          1
'FR000000O0000'          1
'FR00000O00000'          1
'FR0000H000000'          1
'FR00G00000000'          1
'FR000000K0000'  

CONCLUSION POINT 1

Le fichier contient 15 codes distincts dans la colonne pays_declare. 

Pour illustrer la diversité des formats, nous avons étudié le code FR qui représente 974 lignes.

En remplaçant chaque chiffre par 0 et en conservant les autres caractères, nous obtenons 50 formes distinctes, marqueurs d’absence inclus.

Ces formes montrent des différences de longueur, de ponctuation, d’espacement et de casse. Elles ne constituent pas 50 formats valides de TVA.

Point 2 Quelle proportion de lignes porte des caractères qui n'ont rien à faire dans un numéro de TVA ?

In [25]:
tva = df["numero_tva"]

# Réunir tous les caractères présents dans les valeurs brutes
caracteres = sorted(set("".join(tva.dropna())))

print(caracteres)

[' ', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'X', 'Z', 'b', 'd', 'e', 'f', 'g', 'i', 'k', 'l', 'n', 'p', 'q', 'r', 's', 't', 'u', 'x', 'z']


In [26]:
# Repérer les lignes contenant au moins un caractère à retirer
masque_caracteres = tva.str.upper().str.contains(
    r"[^A-Z0-9]", regex=True, na=False
)

# Compter chaque ligne concernée une seule fois
print("Nombre de lignes concernées :", masque_caracteres.sum())

Nombre de lignes concernées : 1861


1 861 lignes concernées. 
Maintenant il faut calculer leur proportion :

In [27]:
pourcentage = masque_caracteres.sum() / len(df) * 100

print(f"Proportion de lignes concernées : {pourcentage:.2f} %")

Proportion de lignes concernées : 18.61 %


1 861 lignes sur 10 000, soit 18,61 %, contiennent au moins un caractère que la fonction normaliser() supprimerait : espace, tiret, point ou barre oblique. Chaque ligne est comptée une seule fois, marqueurs d’absence inclus. La présence de ces caractères ne signifie pas que le numéro de TVA est invalide.

Point 3 Combien de valeurs vides, et sous combien de formes différentes le vide se présente-t-il ?

In [28]:
tva = df["numero_tva"]

print("Valeurs manquantes détectées par pandas :", tva.isna().sum())

# Repérer les textes vides ou composés uniquement d'espaces
masque_texte_vide = tva.str.strip().eq("").fillna(False)

print("Textes vides ou composés d'espaces :", masque_texte_vide.sum())

# Distinguer les formes brutes : '' et ' ', par exemple
display(tva[masque_texte_vide].map(repr).value_counts())

Valeurs manquantes détectées par pandas : 0
Textes vides ou composés d'espaces : 119


numero_tva
''     60
' '    59
Name: count, dtype: int64

Il reste à repérer les marqueurs textuels d’absence comme N/A, null et -.
Il faut bien distinguer 6 formes d’absence au total observées précédemment :
- '' : chaîne vide ;
- ' ' : un espace ;
- '-' ;
- 'N/A' ;
- 'null' ;
- 'NU.LL' : marqueur probable d’absence.
Les deux premières sont déjà comptées dans les 119 lignes. La prochaine étape consiste à compter les quatre autres, en conservant leurs formes brutes et sans modifier les données.


In [29]:
# Repérer les marqueurs textuels sans modifier les données
masque_marqueurs = tva.isin(["-", "N/A", "null", "NU.LL"])

display(tva[masque_marqueurs].map(repr).value_counts())

print("Total des marqueurs textuels :", masque_marqueurs.sum())

numero_tva
'-'        55
'N/A'      48
'null'     38
'NU.LL'     1
Name: count, dtype: int64

Total des marqueurs textuels : 142


La première cellule repère 119 textes vides ou composés d’espaces. La seconde utilise isin() pour rechercher exactement les quatre marqueurs indiqués : elle en trouve 142.

map(repr) rend les espaces et les chaînes vides visibles ; value_counts() compte chaque forme.

Pourquoi pandas affiche-t-il 0 valeur manquante ? Parce que, dans les données actuellement chargées, ces valeurs sont des textes, pas des valeurs manquantes reconnues par isna().

On peut donc conclure : 261 lignes sur 10 000, soit 2,61 %, représentent une absence sous six formes différentes, en précisant que NU.LL est considéré comme un marqueur probable d’absence.

Point 4 Le pays déclaré dans le CRM correspond-il toujours au préfixe du numéro ?

On commence par afficher les deux informations côte à côte, sans corriger les numéros. Le préfixe est repéré par deux lettres au début du texte, après d’éventuels espaces.

In [30]:
# Extraire le préfixe sans modifier la colonne d'origine
prefixe_observe = df["numero_tva"].str.extract(
    r"^\s*([A-Za-z]{2})",
    expand=False
)

apercu_pays = df[["pays_declare", "numero_tva"]].copy()
apercu_pays["prefixe_observe"] = prefixe_observe

display(apercu_pays.head(20))

,pays_declare,numero_tva,prefixe_observe
0,DK,DK73224645,DK
1,PT,PT787327239,PT
2,LU,LU71043234,LU
3,SE,SE.751298146001,SE
4,NL,NL505862176B88,NL
5,DK,DK61530843,DK
6,NL,NL505862176B88,NL
7,SE,SE.314125231601,SE
8,DK,DK48646344,DK
9,BE,BE1997422119,BE


In [31]:
# Exclure les absences et les lignes sans préfixe
masque_absence = masque_texte_vide | masque_marqueurs

comparaison = apercu_pays[
    ~masque_absence & prefixe_observe.notna()
]

# Comparer les codes en ignorant la casse
identiques = (
    comparaison["pays_declare"].str.upper()
    == comparaison["prefixe_observe"].str.upper()
)

print("Codes identiques :", identiques.sum())
print("Codes différents :", (~identiques).sum())
print("Sans préfixe :", (~masque_absence & prefixe_observe.isna()).sum())
print("Absences :", masque_absence.sum())

Codes identiques : 9207
Codes différents : 0
Sans préfixe : 532
Absences : 261


Sur les 9 207 lignes comparables, le pays déclaré correspond toujours au préfixe, en ignorant les différences de casse.

Pour les 793 autres lignes — 532 sans préfixe repérable et 261 absences — la comparaison n’est pas possible.

Attention : une correspondance entre les codes ne garantit pas la validité du numéro de TVA.

Le CSV et l'Excel contiennent-ils vraiment la même chose une fois chargés ? Lequel prenez-vous comme source de vérité, et pourquoi ?

In [32]:
csv = pd.read_csv(
    dossier_data / "numeros_tva.csv",
    dtype=str,
    keep_default_na=False
)

excel = pd.read_excel(
    dossier_data / "numeros_tva.xlsx",
    dtype=str,
    keep_default_na=False
)

print("Dimensions CSV :", csv.shape)
print("Dimensions Excel :", excel.shape)

print("Mêmes colonnes :", csv.columns.equals(excel.columns))
print("Contenus identiques :", csv.equals(excel))

Dimensions CSV : (10000, 6)
Dimensions Excel : (10000, 6)
Mêmes colonnes : True
Contenus identiques : True


Le CSV et l’Excel produisent des tableaux identiques lorsqu’ils sont chargés comme du texte, sans conversion automatique des marqueurs d’absence. Nous retenons le CSV comme source de référence pour la suite : il est directement lisible en texte et facilement exploitable avec pandas. 
Ce choix est pratique.

Point 6 Les canaux de saisie et les périodes de saisie se ressemblent-ils, ou certains sortent-ils du lot ?

On commence par compter les lignes par canal et par année.

In [33]:
# Nombre de lignes par canal
print(df["source_saisie"].value_counts())

# Nombre de lignes par année
annees = pd.to_datetime(df["date_saisie"], errors="coerce").dt.year
print(annees.value_counts().sort_index())

source_saisie
reprise_erp           2071
crm                   2019
portail_client        1990
saisie_manuelle       1976
import_fournisseur    1944
Name: count, dtype: int64
date_saisie
2024    5760
2025    4240
Name: count, dtype: int64


On compare maintenant les pourcentages d'absences et de caractères à retirer, avec les variables déjà créées. 

In [34]:
# Réunir les indicateurs dans un tableau séparé
qualite = pd.DataFrame({
    "canal": df["source_saisie"],
    "annee": annees,
    "absence": masque_absence,
    "caracteres_a_retirer": masque_caracteres
})

print("Pourcentages par canal :")
display(
    (qualite.groupby("canal")[["absence", "caracteres_a_retirer"]]
     .mean() * 100).round(2)
)

print("Pourcentages par année :")
display(
    (qualite.groupby("annee")[["absence", "caracteres_a_retirer"]]
     .mean() * 100).round(2)
)


Pourcentages par canal :


,absence,caracteres_a_retirer
canal,,
crm,2.82,19.07
import_fournisseur,2.88,19.55
portail_client,2.36,17.99
reprise_erp,2.85,18.78
saisie_manuelle,2.13,17.66


Pourcentages par année :


,absence,caracteres_a_retirer
annee,,
2024,2.48,19.08
2025,2.78,17.97


Les canaux présentent des taux proches :
- Absences : de 2,13 % en saisie manuelle à 2,88 % pour les imports fournisseurs.
- Caractères à retirer : de 17,66 % en saisie manuelle à 19,55 % pour les imports fournisseurs.
Entre 2024 et 2025, les absences passent de 2,48 % à 2,78 %, tandis que les caractères à retirer passent de 19,08 % à 17,97 %.

Sur les deux indicateurs étudiés, les canaux et les années présentent des proportions relativement proches. Les imports fournisseurs affichent les taux les plus élevés, mais les écarts restent modestes : aucun groupe ne se démarque fortement. 

Nous pouvons pousser la réflexion plus loin en faisant une analyse par mois et non par an pour vérifier s'il y a des anomalies

Il faut commencer par compter les lignes par mois avant de comparer les indicateurs

In [35]:
mois = pd.to_datetime(df["date_saisie"]).dt.to_period("M")

print(mois.value_counts().sort_index())

date_saisie
2024-01    461
2024-02    446
2024-03    504
2024-04    510
2024-05    489
2024-06    503
2024-07    469
2024-08    489
2024-09    443
2024-10    458
2024-11    494
2024-12    494
2025-01    497
2025-02    428
2025-03    456
2025-04    465
2025-05    457
2025-06    468
2025-07    500
2025-08    508
2025-09    445
2025-10     16
Freq: M, Name: count, dtype: int64


Les volumes vont de 428 à 510 lignes par mois, sauf octobre 2025, avec seulement 16 lignes. Avant d’interpréter cette baisse, vérifions si ce dernier mois est couvert en entier.

In [36]:
dates = pd.to_datetime(df["date_saisie"])

print("Première date :", dates.min())
print("Dernière date :", dates.max())

Première date : 2024-01-01 00:00:00
Dernière date : 2025-10-01 00:00:00


Effectivement le mois d'octobre 2025 couvre qu'une seule journée. Il conviendra de faire attention dans les prochaine analyse. 

Analysons les canaux mois par mois pour voir s'il y a des anomalies

On commence par compter les lignes de chaque canal de saisie, mois par mois.

In [37]:
display(pd.crosstab(mois, df["source_saisie"]))

source_saisie,crm,import_fournisseur,portail_client,reprise_erp,saisie_manuelle
date_saisie,,,,,
2024-01,85,88,104,104,80
2024-02,103,88,83,84,88
2024-03,111,93,100,107,93
2024-04,116,103,88,103,100
2024-05,85,96,104,103,101
2024-06,119,103,89,103,89
2024-07,86,85,87,112,99
2024-08,101,103,100,100,85
2024-09,92,89,74,104,84


Chaque ligne représente un mois et chaque colonne un canal. Ce tableau permettre de voir si certains canaux deviennent plus ou moins présents selon les périodes. 

Ce que nous pouvons observer : 
Les cinq canaux restent présents chaque mois et leurs volumes sont globalement proches, avec quelques variations ponctuelles.
- Novembre 2024 est très équilibré : chaque canal compte entre 97 et 102 lignes.
- Février 2025 présente davantage d’écart : le portail client représente 106 lignes sur 428 (24,8 %), contre 69 pour les imports fournisseurs (16,1 %).
- Septembre 2025 montre un autre contraste : 111 saisies manuelles (24,9 %) contre 71 lignes CRM (16,0 %).
- Octobre 2025 doit être interprété séparément : il ne couvre qu’une journée et compte 16 lignes. Ses faibles volumes ne prouvent pas une baisse d’activité.
Ces volumes seuls ne permettent pas de conclure à une anomalie de qualité.